# Entrenamiento del modelo final — MercadoLibre: ¿nuevo o usado?

Entrena y guarda en `modelo_final/` todo lo que usa `aplicar_modelo_final.ipynb` para generar el envío
`ensamble_fable_v9.csv` (accuracy 0.95188 en el leaderboard privado).

**Modelo final** (detalle en el README):

1. **Features** de la publicación, del vendedor y de la categoría + features de nivel 1 que usan la etiqueta
   (target encoding, modelos de texto, vecinos), siempre calculadas *out-of-fold*.
2. **Ensamble**: 0,5 × TabPFN + 0,5 × promedio de LightGBM, XGBoost y CatBoost (3 semillas cada uno).
3. **Combinación con un LLM** en los tres tramos donde el ensamble sabe poco del vendedor: regresión logística sobre
   `[logit(ensamble), logit(LLM)]`, ajustada por tramo.

**Entradas que no se recalculan acá** (ver README):

- `datos/`: `train_data.jsonlines` y `test_data.jsonlines` de la competencia, y `categories.json` (API pública de MercadoLibre).
- `gpu_kaggle/salidas/`: predicciones de los tres transformers y de TabPFN, que necesitan GPU y corrieron en Kaggle con los notebooks de `gpu_kaggle/`.
- `llm/puntajes/`: puntajes de Claude, generados con `llm/llm_claude.py`. Las respuestas de un LLM no son deterministas, así que quedan guardadas.
- `datos/embeddings_e5/`: embeddings `multilingual-e5-small` de los títulos. Recalcularlos en otra máquina da diferencias de 1e-7 que alcanzan para mover un puñado de predicciones (ver README), así que se guardan los que usó el modelo enviado. Si faltan, se recalculan.

Tiempo aproximado en CPU: 3 horas (40 min de features y 2 h 20 min de boostings; CatBoost es el más lento).

In [1]:
import os, glob, json, time, joblib, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import features as F
warnings.filterwarnings("ignore")
os.makedirs(F.CACHE, exist_ok=True); os.makedirs("modelo_final", exist_ok=True)

## 1. Features

**Features de fila** y **agregados no supervisados** por vendedor y categoría (`features.py`). Se calculan sobre train+test juntos porque no usan la etiqueta.

In [2]:
train, test = F.load("train_data.jsonlines"), F.load("test_data.jsonlines")
y = (train[F.TARGET] == "new").astype(int).values
X_train, X_test = F.build(train, test)
np.save(f"{F.CACHE}/y.npy", y)
test[["id"]].to_csv(f"{F.CACHE}/test_ids.csv", index=False)
print(X_train.shape, X_test.shape, "proporción new:", y.mean().round(3))

(70000, 173) (30000, 173) proporción new: 0.537


Texto del título y la garantía: TF-IDF de palabras y caracteres (el vocabulario se ajusta con train+test, sin etiquetas) y embeddings `multilingual-e5-small`.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
txt = np.concatenate([F.text_field(train), F.text_field(test)])
W = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(txt)
Ch = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=400_000).fit_transform(txt)
T = sp.hstack([W, Ch]).tocsr()
T_train, T_test = T[:len(train)], T[len(train):]

# Embeddings guardados (los del modelo enviado), partidos en 4 archivos por el límite de tamaño de GitHub
partes = sorted(glob.glob("datos/embeddings_e5/parte*.npy"))
if partes:
    E = np.concatenate([np.load(p) for p in partes])
else:
    print("Recalculando embeddings: el resultado puede diferir en unas pocas predicciones del archivo enviado")
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu"); st.max_seq_length = 48
    titles = ["query: " + t for t in pd.concat([train["title"], test["title"]]).fillna("")]
    E = st.encode(titles, batch_size=128, normalize_embeddings=True, show_progress_bar=False).astype("float32")
E_train, E_test = E[:len(train)], E[len(train):]
T.shape, E.shape

((100000, 254377), (100000, 384))

**Features de nivel 1** (usan la etiqueta): target encoding suavizado de vendedor, categoría, ciudad, garantía y combinaciones vendedor × atributo; regresiones logísticas sobre TF-IDF y sobre embeddings; kNN de títulos; vecinos de título dentro del mismo vendedor.

Hay una versión por fold de validación, ajustada solo con la parte de entrenamiento del fold, que alimenta las predicciones *out-of-fold* para ajustar el ensamble y la combinación con el LLM. Hay otra final, ajustada con todo train y aplicada a test. Las columnas de los transformers (`txt_bert`, `txt_xlmr`, `txt_bertin`) se agregan al leer, con `F.load_fold` / `F.load_full`.

In [4]:
for f, (a, b) in enumerate(F.cv_folds(y)):
    A, B = F.level1(X_train.iloc[a], y[a], X_train.iloc[b], T_train[a], T_train[b], E_train[a], E_train[b])
    A.to_parquet(f"{F.CACHE}/fold{f}_tr.parquet"); B.to_parquet(f"{F.CACHE}/fold{f}_va.parquet")
A, B = F.level1(X_train, y, X_test, T_train, T_test, E_train, E_test)
A.to_parquet(f"{F.CACHE}/full_tr.parquet"); B.to_parquet(f"{F.CACHE}/full_te.parquet")
A, B = F.load_full()
print("features del modelo:", A.shape[1], "| train", A.shape, "| test", B.shape)

features del modelo: 200 | train (70000, 200) | test (30000, 200)


## 2. Boostings

Mismos 5 folds estratificados para todos los modelos (`F.cv_folds`). Cada boosting se entrena:

- por fold, para tener su predicción *out-of-fold* (entrada del ensamble y de la combinación con el LLM);
- con todo train: es el modelo que se guarda y se aplica a test.

Se promedian 3 semillas para reducir la varianza.

In [5]:
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
SEEDS = [42, 43, 44]

BOOSTINGS = {
    "lightgbm": lambda s: lgb.LGBMClassifier(objective="binary", learning_rate=0.02, n_estimators=1200, num_leaves=63,
                                             min_child_samples=10, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                                             reg_lambda=5, n_jobs=6, verbose=-1, random_state=s),
    "xgboost": lambda s: xgb.XGBClassifier(learning_rate=0.03, n_estimators=1200, max_depth=9, min_child_weight=2,
                                           colsample_bytree=0.3, subsample=0.8, reg_lambda=3, tree_method="hist",
                                           enable_categorical=True, max_cat_to_onehot=64, n_jobs=6, random_state=s),
    "catboost": lambda s: CatBoostClassifier(iterations=800, learning_rate=0.08, depth=8, l2_leaf_reg=3, max_ctr_complexity=1,
                                             one_hot_max_size=32, thread_count=8, verbose=0, allow_writing_files=False,
                                             random_seed=s),
}


def str_cats(D):
    # CatBoost recibe las categoricas de baja cardinalidad como texto
    D = D.copy(); cats = [c for c in D.columns if str(D[c].dtype) == "category"]
    D[cats] = D[cats].astype(str)
    return D, cats


def fit(name, A, ya):
    if name == "catboost":
        A, cats = str_cats(A)
        return [BOOSTINGS[name](s).fit(A, ya, cat_features=cats) for s in SEEDS]
    return [BOOSTINGS[name](s).fit(A, ya) for s in SEEDS]


def predict(name, models, B):
    if name == "catboost":
        B, _ = str_cats(B)
    return np.mean([m.predict_proba(B)[:, 1] for m in models], axis=0)


def save(name, models):
    for s, m in zip(SEEDS, models):
        if name == "lightgbm": m.booster_.save_model(f"modelo_final/lightgbm_seed{s}.txt")
        if name == "xgboost": m.save_model(f"modelo_final/xgboost_seed{s}.json")
        if name == "catboost": m.save_model(f"modelo_final/catboost_seed{s}.cbm")

In [6]:
folds = F.cv_folds(y)
oof = {}
for name in BOOSTINGS:
    t0 = time.time(); oof[name] = np.zeros(len(y))
    for f, (a, b) in enumerate(folds):
        A, B = F.load_fold(f)
        oof[name][b] = predict(name, fit(name, A, y[a]), B)
    A, _ = F.load_full()
    save(name, fit(name, A, y))
    np.save(f"{F.CACHE}/oof_{name}.npy", oof[name])
    print(f"{name}: accuracy CV {((oof[name] > .5) == y).mean():.4f} ({(time.time() - t0) / 60:.0f} min)")

lightgbm: accuracy CV 0.9483 (22 min)


xgboost: accuracy CV 0.9476 (47 min)


catboost: accuracy CV 0.9481 (73 min)


## 3. Ensamble con TabPFN

TabPFN (modelo fundacional tabular, versión 3.5 con 4 estimadores) recibe las mismas features y los mismos folds. Corrió en Kaggle con GPU (`gpu_kaggle/tabpfn.ipynb`): no se entrena, usa las filas de entrenamiento como contexto. Su salida trae la predicción *out-of-fold* y la de test.

El peso 0,5 se eligió con las predicciones *out-of-fold*: la curva es plana entre 0,4 y 0,7.

In [7]:
tabpfn = np.load("gpu_kaggle/salidas/tabpfn.npz")
acc = lambda p, t: ((p > .5) == t).mean()
ens_oof = 0.5 * tabpfn["oof"] + 0.5 * np.mean([oof[n] for n in BOOSTINGS], axis=0)
print(f"TabPFN: accuracy CV {acc(tabpfn['oof'], y):.4f} | ensamble: {acc(ens_oof, y):.4f}")

TabPFN: accuracy CV 0.9474 | ensamble: 0.9491


## 4. Combinación con el LLM por tramos

Claude estima la probabilidad de que la publicación sea nueva leyendo título, categoría, precio, garantía, etc. Aporta donde el ensamble sabe poco del vendedor:

| tramo (historial del vendedor con etiqueta) | LLM | qué ve el LLM |
|---|---|---|
| vendedor nuevo (0 publicaciones) | Claude Opus 5.5 | solo la publicación |
| 1 a 3 publicaciones | Claude Opus 5.5 | solo la publicación |
| mixto: ≥ 4 publicaciones y entre 20% y 80% nuevas | Claude Fable 5.2 | además, sus publicaciones más parecidas con la condición real |

En train, el "historial" se cuenta solo dentro de la parte de entrenamiento del fold de cada fila (lo mismo que ve el ensamble). En test se cuenta en todo train. En cada tramo se ajusta una regresión logística sobre `[logit(ensamble), logit(LLM)]` con las filas de train del tramo que evaluó el LLM.

In [8]:
from sklearn.linear_model import LogisticRegression
lg = lambda q: np.log(np.clip(q, .01, .99) / (1 - np.clip(q, .01, .99)))


def puntajes_llm(version, partes):
    P = {}
    for parte in partes:
        for f in glob.glob(f"llm/puntajes/{version}/{parte}/*.json"):
            P.update({int(k): v / 100 for k, v in json.load(open(f, encoding="utf8"))["p"].items()})
    return P


seller = train["seller_id"].values
hist, pur = np.zeros(len(y)), np.full(len(y), np.nan)
for a, b in folds:
    g = pd.Series(y[a]).groupby(seller[a]).agg(["mean", "size"])
    hist[b] = pd.Series(seller[b]).map(g["size"]).fillna(0).values
    pur[b] = pd.Series(seller[b]).map(g["mean"]).values

TRAMOS = {  # nombre: (filtro sobre historial y fraccion de nuevas, version del LLM, corridas de train)
    "nuevos": (lambda h, p: h == 0, "claude-opus-5-5", ["piloto", "etapa1", "nuevos_extra"]),
    "pocos": (lambda h, p: (h >= 1) & (h <= 3), "claude-opus-5-5", ["piloto", "etapa1"]),
    "mixtos": (lambda h, p: (h >= 4) & (p >= .2) & (p <= .8), "claude-fable-5-2-v3", ["mixtos_train", "mixtos_train2"]),
}
combinacion = {}
for nombre, (filtro, version, partes) in TRAMOS.items():
    P = puntajes_llm(version, partes)
    idx = np.array(sorted(P), dtype=int)
    tr = idx[filtro(hist[idx], pur[idx])]
    Z = np.c_[lg(ens_oof[tr]), lg(np.array([P[i] for i in tr]))]
    m = combinacion[nombre] = LogisticRegression().fit(Z, y[tr])
    print(f"{nombre}: {len(tr)} filas de train | pesos (logit) ensamble {m.coef_[0][0]:.3f}, LLM {m.coef_[0][1]:.3f}, "
          f"intercepto {m.intercept_[0]:.3f}")
joblib.dump(combinacion, "modelo_final/combinacion_llm.joblib")
sorted(os.listdir("modelo_final"))

nuevos: 6457 filas de train | pesos (logit) ensamble 0.757, LLM 0.612, intercepto 0.286
pocos: 1664 filas de train | pesos (logit) ensamble 0.937, LLM 0.515, intercepto 0.138


mixtos: 2000 filas de train | pesos (logit) ensamble 0.811, LLM 0.600, intercepto 0.238


['catboost_seed42.cbm',
 'catboost_seed43.cbm',
 'catboost_seed44.cbm',
 'combinacion_llm.joblib',
 'lightgbm_seed42.txt',
 'lightgbm_seed43.txt',
 'lightgbm_seed44.txt',
 'xgboost_seed42.json',
 'xgboost_seed43.json',
 'xgboost_seed44.json']